# 🐍 Aula 09 Extra - Mutabilidade, Escopo e Funções Avançadas (*args, **kwargs, Argumentos Padrão e Type Hinting)

### Objetivos desta Edição Estendida (Extra)
Esta aula consolida em um único guia completo tudo o que você precisa dominar sobre **gerenciamento de memória**, **escopo de variáveis** e **construção de funções profissionais** em Python:

1. **Mutabilidade vs Imutabilidade**: Compreender como o interpretador aloca objetos na memória (`id()`), diferenciando tipos imutáveis (`int`, `float`, `str`, `tuple`, `bool`) de tipos mutáveis (`list`, `dict`, `set`).
2. **Passagem de Parâmetros e Proteção de Dados**: Dominar o modelo de *Passagem por Atribuição* (*Pass-by-assignment*) e aplicar cópias rasas (`.copy()`, `[:]`) e profundas (`copy.deepcopy()`) para evitar efeitos colaterais indesejados (*side effects*).
3. **Escopo de Variáveis e Regra LEGB**: Compreender a visibilidade e o ciclo de vida de variáveis seguindo a hierarquia **L**ocal $\rightarrow$ **E**nclosing $\rightarrow$ **G**lobal $\rightarrow$ **B**uilt-in.
4. **Shadowing e Uso Consciente de `global`**: Entender o sombreamento de variáveis e os motivos pelos quais a engenharia de software moderna prefere funções puras com retornos explícitos em tupla (`return a, b, c`).
5. **Argumentos Padrão (Default Parameters)**: Declarar valores padrão para parâmetros opcionais e conhecer a armadilha do valor padrão mutável.
6. **Empacotamento e Desempacotamento com `*args` e `**kwargs`**: Construir funções flexíveis capazes de receber qualquer quantidade de argumentos posicionais (tuplas) e nomeados (dicionários).
7. **Tipagem Estática Opcional (Type Hinting)**: Documentar assinaturas de funções com o módulo `typing` (`List`, `Dict`, `Tuple`, `Optional`).
8. **Bateria de Exercícios Práticos Dirigidos**: 8 desafios completos e comentados cobrindo todos os tópicos no contexto dos sistemas do submarino CIAA-LPS.

## 1. Revisão de Mutabilidade e Imutabilidade na Memória

Em Python, **tudo é um objeto**. Cada objeto criado no programa possui três características fundamentais gerenciadas pelo interpretador:
1. **Identidade (`id`)**: O endereço único de memória onde o objeto reside.
2. **Tipo (`type`)**: A classe do objeto (ex: `int`, `str`, `list`), que determina quais operações são permitidas.
3. **Valor**: O dado contido no objeto.

### O que significa ser Mutável ou Imutável?

* **Objetos Imutáveis**: Uma vez criados na memória, **seu conteúdo nunca pode ser alterado**. Se tentarmos "modificar" uma string, número ou tupla, o Python obrigatoriamente criará um **novo objeto** em um novo endereço de memória.
* **Objetos Mutáveis**: Podem ter seu conteúdo interno alterado (adicionar, remover ou modificar elementos) **mantendo o mesmo endereço de memória (`id`)**.

| Categoria | Tipos de Dados em Python | Pode alterar elementos internos? | O que acontece ao modificar? |
| :--- | :--- | :--- | :--- |
| **Imutáveis** | `int`, `float`, `str`, `tuple`, `bool`, `bytes` | ❌ Não | Cria um novo objeto na memória com novo `id()` |
| **Mutáveis** | `list`, `dict`, `set`, `bytearray` | ✅ Sim | Altera o objeto *in-place* no mesmo `id()` |

> 🔍 **Operador `is` vs Operador `==`**:
> * `a == b`: Compara se os **valores** de `a` e `b` são iguais.
> * `a is b`: Compara se `a` e `b` apontam para o **mesmo endereço de memória** (`id(a) == id(b)`).

In [1]:
# --- DEMONSTRAÇÃO 1: Imutabilidade de Números e Strings ---
profundidade = 100
print(f"Profundidade inicial: {profundidade} | Endereço id: {id(profundidade)}")

# Ao somar 50, criamos um NOVO objeto int na memória
profundidade = profundidade + 50
print(f"Nova profundidade:   {profundidade} | Novo endereço id: {id(profundidade)}")

print("-" * 50)

# --- DEMONSTRAÇÃO 2: Mutabilidade de Listas ---
sensores = ["Sonar", "Pressão"]
print(f"Lista inicial: {sensores} | Endereço id: {id(sensores)}")

# Ao usar .append(), alteramos a lista no MESMO endereço de memória
sensores.append("Temperatura")
print(f"Lista após append: {sensores} | Mesmo endereço id: {id(sensores)}")

Profundidade inicial: 100 | Endereço id: 4332817744
Nova profundidade:   150 | Novo endereço id: 4332819344
--------------------------------------------------
Lista inicial: ['Sonar', 'Pressão'] | Endereço id: 4393022464
Lista após append: ['Sonar', 'Pressão', 'Temperatura'] | Mesmo endereço id: 4393022464


## 2. Passagem de Argumentos para Funções e Efeitos Colaterais

Em Python, a passagem de argumentos para funções é feita por **Atribuição de Referência de Objeto** (*Pass-by-object-reference* ou *Pass-by-assignment*):

1. Se você passar um **objeto imutável** (como `int` ou `tuple`), a função recebe a referência. Qualquer tentativa de reatribuir o valor criará uma nova variável local dentro da função, mantendo a variável original externa intocada.
2. Se você passar um **objeto mutável** (como `list` ou `dict`), a função recebe a referência direta para o mesmo endereço de memória. Se a função alterar a lista (ex: `.append()`, `.pop()`, `d[k] = v`), **a variável original fora da função será alterada!** Esse comportamento é chamado de **Efeito Colateral (*Side Effect*)**.

### Como se proteger contra Efeitos Colaterais Indesejados?

Quando uma função precisa processar uma lista ou dicionário sem modificar o dado original que o chamador enviou, devemos criar uma **cópia**:
* **Cópia Rasa (*Shallow Copy*)**:
  * Listas: `nova_lista = lista_original.copy()` ou `nova_lista = list(lista_original)` ou `nova_lista = lista_original[:]`
  * Dicionários: `novo_dict = dict_original.copy()` ou `novo_dict = dict(dict_original)`
* **Cópia Profunda (*Deep Copy*)** (necessária quando há listas/dicionários aninhados dentro da estrutura):
  * `import copy; copia_profunda = copy.deepcopy(estrutura_aninhada)`

In [2]:
# --- EXEMPLO: Função com Efeito Colateral Indesejado vs Função Segura ---

# 1. Função Insegura (Modifica a lista do chamador diretamente na memória)
def adicionar_alvo_inseguro(lista_alvos, novo_alvo):
    lista_alvos.append(novo_alvo) # Altera o objeto original!
    #return lista_alvos # ["ALVO-1", "ALVO-2", ALVO-3]

radar_base = ["ALVO-1", "ALVO-2"]
print("Radar antes da função insegura:", radar_base)
adicionar_alvo_inseguro(radar_base, "ALVO-3")
print("Radar APÓS a função insegura (Foi alterado!):", radar_base)


Radar antes da função insegura: ['ALVO-1', 'ALVO-2']
Radar APÓS a função insegura (Foi alterado!): ['ALVO-1', 'ALVO-2', 'ALVO-3']


In [3]:

print("-" * 50)

# 2. Função Segura (Cria uma cópia e não altera a lista original)
def adicionar_alvo_seguro(lista_alvos, novo_alvo):
    lista_copia = lista_alvos.copy() # Cria uma nova lista na memória
    lista_copia.append(novo_alvo)
    return lista_copia

radar_oficial = ["ALVO-A", "ALVO-B"]
print("Radar oficial antes:", radar_oficial)
radar_simulado = adicionar_alvo_seguro(radar_oficial, "ALVO-C")
print("Radar oficial pós-função (Protegido!):", radar_oficial)
print("Novo radar retornado pela função:", radar_simulado)

--------------------------------------------------
Radar oficial antes: ['ALVO-A', 'ALVO-B']
Radar oficial pós-função (Protegido!): ['ALVO-A', 'ALVO-B']
Novo radar retornado pela função: ['ALVO-A', 'ALVO-B', 'ALVO-C']


## 3. Escopo de Variáveis em Python (Local vs Global)

O **Escopo** (*Scope*) de uma variável define **onde** no código essa variável pode ser acessada e qual é o seu **tempo de vida** durante a execução do programa.

### A Regra LEGB de Busca de Nomes
Quando você utiliza uma variável no Python, o interpretador busca o nome nessa ordem exata:

```text
 ┌────────────────────────────────────────────────────────┐
 │  L - Local:      Definida dentro da função atual       │
 │       ↓                                                │
 │  E - Enclosing:  Definida em funções aninhadas/externas│
 │       ↓                                                │
 │  G - Global:     Definida no corpo principal do módulo │
 │       ↓                                                │
 │  B - Built-in:   Nomes nativos do Python (len, min...) │
 └────────────────────────────────────────────────────────┘
```

### Variável Local vs Variável Global

1. **Variável Local**:
   * É criada no momento em que a função é executada.
   * Existe apenas na memória **enquanto a função está rodando**.
   * É destruída automaticamente quando a função atinge o `return` ou termina.
   * **Não pode ser acessada de fora da função** (tentar acessá-la causa `NameError`).

2. **Variável Global**:
   * É criada fora de qualquer função, no nível principal do arquivo/célula do notebook.
   * Permanece viva durante toda a execução do programa.
   * Pode ser **lida** livremente por qualquer função do código.

In [19]:
# --- DEMONSTRAÇÃO DE ESCOPO GLOBAL E LOCAL ---

# Variável no Escopo Global (Visível em todo o script)
NOME_SUBMARINO = "CIAA-LPS-Almirante"
PRESSAO_MAXIMA_CASCO = 400.0

#def calcular_margem_seguranca(profundidade_atual, pressao_maxima_casco):]
def calcular_margem_seguranca(profundidade_atual):

    # profundidade_atual é LOCAL (parâmetro)
    # fator_conversao é LOCAL (criada dentro da função)
    fator_conversao = 10.0 # Cada 10m de água equivalem a aprox 1 atm extra
    pressao_estimada = profundidade_atual / fator_conversao
    
    # A função consegue LER a variável GLOBAL PRESSAO_MAXIMA_CASCO diretamente:
    margem = PRESSAO_MAXIMA_CASCO - pressao_estimada
    #margem = pressao_maxima_casco - pressao_estimada
    return margem

# Executando a função
resultado_margem = calcular_margem_seguranca(150.0)
print(f"Submarino: {NOME_SUBMARINO}")
print(f"Margem de Segurança a 150m: {resultado_margem} atm")

# Tentativa de acessar variável local fora da função:
#print(fator_conversao)
#try:
#    print(fator_conversao)
#except NameError as erro:
#    print(f"❌ Erro esperado ao tentar acessar variável local: {erro}")

Submarino: CIAA-LPS-Almirante
Margem de Segurança a 150m: 385.0 atm


## 4. Shadowing, a Palavra-Chave `global` e Boas Práticas

### O Fenômeno de *Shadowing* (Sombreamento)
Se você definir uma variável dentro de uma função com o **mesmo nome** de uma variável global, o Python criará uma variável **local** independente. A variável local "faz sombra" à variável global dentro da função, mas a variável global externa **não é modificada**.

### A Armadilha do `UnboundLocalError`
Se uma função tentar ler uma variável global e, mais abaixo no mesmo corpo da função, tentar atribuir um valor a uma variável com o mesmo nome, o Python assume que ela é local para a função inteira. Ao tentar lê-la antes da linha de atribuição, ocorre o erro `UnboundLocalError: local variable referenced before assignment`.

### A Palavra-Chave `global`
Se você realmente precisar **alterar** o valor de uma variável global a partir de dentro de uma função, você deve declarar `global nome_variavel` antes de atribuir o novo valor.

> ⚠️ **Boas Práticas de Engenharia de Software**:
> O uso da palavra-chave `global` para manter ou alterar o estado do sistema deve ser **evitado**. Funções que alteram variáveis globais são difíceis de testar, criam acoplamento e produzem bugs difíceis de rastrear. 
> 
> **A Abordagem Recomendada**: Escreva **funções puras** que recebem todos os dados de que precisam via parâmetros e retornam os resultados calculados via `return` (usando tuplas `return a, b, c` quando houver múltiplos valores).

In [9]:
# --- DEMONSTRAÇÃO 1: Shadowing (Sombreamento de Variável) ---
status_missao = "EM_PATRULHA" # str # Global @A

def testar_shadowing():
    global status_missao
    status_missao = "EM_DOCAGEM"  # @B Cria variável LOCAL com mesmo nome
    print("Dentro da função (Local):", status_missao)


testar_shadowing()
print("Fora da função (Global permaneceu inalterada):", status_missao)

print("-" * 50)

Dentro da função (Local): EM_DOCAGEM
Fora da função (Global permaneceu inalterada): EM_DOCAGEM
--------------------------------------------------


In [11]:
VARIAVEL_GLOBAL = 1



# --- DEMONSTRAÇÃO 2: Uso da palavra-chave global ---
CONTADOR_ALERTAS = 0 # Global , imutavel

def registrar_alerta_global():
    global CONTADOR_ALERTAS # Avisa ao Python para alterar a variável externa
    CONTADOR_ALERTAS += 1

registrar_alerta_global()
registrar_alerta_global()
print("Total de alertas acumulados no escopo global:", CONTADOR_ALERTAS)


Total de alertas acumulados no escopo global: 2


## 5. Argumentos Padrão (Default Parameters)

Em Python, podemos definir **valores padrões** para os parâmetros de uma função. Isso torna esses parâmetros **opcionais** durante a chamada: caso o chamador não forneça um valor, o valor padrão pré-definido será automaticamente utilizado.

> ⚠️ **Regra Sintática Fundamental**: Todos os parâmetros com valor padrão **devem vir obrigatoriamente depois** de todos os parâmetros obrigatórios (sem valor padrão):
> * ✅ `def configurar_sensor(sensor_id, intervalo=1.0, ativo=True):`
> * ❌ `def configurar_sensor(intervalo=1.0, sensor_id):` $\rightarrow$ *SyntaxError: non-default argument follows default argument*

> 💡 **Atenção à Armadilha de Padrões Mutáveis**: Nunca utilize coleções mutáveis (`[]` ou `{}`) como valor padrão! Em vez disso, use `None` como valor sentinela:
> ```python
> # Incorreto: def adicionar_log(msg, historico=[]):
> # Correto:
> def adicionar_log(msg, historico=None):
>     if historico is None:
>         historico = []
>     historico.append(msg)
>     return historico
> ```

In [ ]:
# 3. Argumentos Padrão em Ação
def enviar_alerta(alvo, nivel="ALERTA"):
    # Se o nível não for especificado, assume "ALERTA"
    print(f"[{nivel}] Alvo {alvo} detectado!")

# Chamada omitindo o argumento padrão
enviar_alerta("Submarino Hostil")

# Chamada fornecendo todos os argumentos
enviar_alerta("Cardume de Peixes", nivel="INFO")

## 6. Empacotamento e Desempacotamento com `*args` e `**kwargs`

Muitas vezes precisamos criar funções que recebem uma quantidade **indefinida ou flexível** de parâmetros:

1. **`*args` (Argumentos Posicionais Variáveis)**:
   * O asterisco `*` empacota todos os argumentos posicionais extras em uma **tupla** (`tuple`).
   * Ideal para funções que operam sobre sequências arbitrárias (ex: calcular a média de 2, 5 ou 100 números).

2. **`**kwargs` (Argumentos Nomeados Variáveis - Keyword Arguments)**:
   * Os dois asteriscos `**` empacotam todos os argumentos nomeados extras em um **dicionário** (`dict`).
   * Ideal para funções que recebem configurações dinâmicas de múltiplos subsistemas com formato `chave=valor`.

```python
def funcao_flexivel(obrigatorio, *args, default_param="teste", **kwargs):
    print("Obrigatório:", obrigatorio)
    print("Args (Tupla):", args)
    print("Default:", default_param)
    print("Kwargs (Dicionário):", kwargs)
```

In [ ]:
# 4. Uso de *args (Empacotamento Posicional)
def listar_contatos(*alvos):
    # 'alvos' é recebido como uma tupla
    print("Tipo de 'alvos':", type(alvos))
    print("Coleção de alvos captados:")
    for alvo in alvos:
        print(f"- {alvo}")

# Podemos passar 1, 3 ou nenhum argumento
listar_contatos("ALVO-1")
print("-" * 30)
listar_contatos("ALVO-A", "ALVO-B", "ALVO-C")

In [ ]:
# 5. Uso de **kwargs (Empacotamento Nomeado)
def detalhar_status_reator(reator_id, **detalhes):
    # 'detalhes' é recebido como um dicionário
    print(f"Status do Reator: {reator_id}")
    for chave, valor in detalhes.items():
        print(f"- {chave.upper()}: {valor}")

# Passamos chaves-valores customizadas na chamada
detalhar_status_reator("REATOR-PRINCIPAL", temperatura=285.5, pressao=15.2, status="Normal")

## 7. Tipagem de Dados em Funções (Type Hinting)

O Python é uma linguagem de tipagem dinâmica, o que significa que não precisamos declarar obrigatoriamente o tipo de uma variável ao criá-la. No entanto, em sistemas complexos (como a navegação do submarino), indicar quais tipos de dados uma função espera receber e retornar torna o código muito mais fácil de ler, documentar e depurar. 

Essa funcionalidade é chamada de **Type Hinting** (indicação de tipo).

### Tipos Simples vs. Estruturas Complexas

Para tipos de dados simples, usamos os próprios tipos built-in:
* `nome: str` (para texto)
* `valor: float` ou `quantidade: int`
* `ativo: bool`

Para estruturas de dados como listas, dicionários e tuplas, importamos utilitários adicionais do módulo nativo `typing` (com a primeira letra maiúscula):
* `List[tipo]` ou `list[tipo]`: Uma lista contendo elementos de determinado tipo.
* `Dict[tipo_chave, tipo_valor]` ou `dict[tipo_chave, tipo_valor]`: Um dicionário associando chaves e valores.
* `Tuple[tipo_1, tipo_2]` ou `tuple[tipo_1, tipo_2]`: Uma tupla contendo elementos nas posições especificadas.

*Nota: A partir do Python 3.9, você pode usar os tipos em letras minúsculas (`list`, `dict`, `tuple`) diretamente sem precisar importar o módulo `typing`, mas em projetos anteriores a importação com letras maiúsculas (`List`, `Dict`, `Tuple`) é o padrão.*

In [ ]:
# 6. Exemplo de Type Hinting
from typing import List, Dict, Tuple

# Esta função indica que recebe uma lista de floats, um dicionário com chave str e valor str,
# e retorna uma tupla contendo um float e um booleano.
def processar_telemetria(leituras: List[float], limites: Dict[str, float]) -> Tuple[float, bool]:
    soma = sum(leituras)
    limite_critico = limites.get("critico", 100.0)
    
    # Verifica se a soma ultrapassa o limite crítico
    sobrecarga = soma > limite_critico
    return soma, sobrecarga

# Utilizando a função normalmente
leituras_bateria = [35.5, 42.0, 15.0]
config_limites = {"alerta": 80.0, "critico": 90.0}

total, perigo = processar_telemetria(leituras_bateria, config_limites)
print(f"Total lido: {total} | Em perigo? {perigo}")

## Tabela Comparativa de Tipos de Parâmetros

| Parâmetro | Sintaxe na Função | Como os Dados São Recebidos | Exemplo de Chamada | Caso de Uso Principal |
| :--- | :--- | :--- | :--- | :--- |
| **Posicional Obrigatório** | `def f(x):` | Valor individual | `f(10)` | Parâmetros essenciais. |
| **Argumento Padrão (Default)**| `def f(x=5):` | Valor individual | `f()` ou `f(10)` | Parâmetros opcionais e configurações padrão. |
| **Argumentos Posicionais Variáveis (`*args`)** | `def f(*args):` | Tupla (`tuple`) | `f(1, 2, 3)` | Quando a função pode receber infinitos valores isolados (ex: somar tudo). |
| **Argumentos Nomeados Variáveis (`**kwargs`)**| `def f(**kwargs):`| Dicionário (`dict`)| `f(a=1, b=2)` | Passagem de parâmetros de configuração opcionais nomeados. |

## 8. Exercícios Práticos Dirigidos

Agora teste seu domínio completo sobre **Mutabilidade, Imutabilidade, Escopo e Funções Avançadas** com os 8 desafios práticos abaixo!

A bateria está estruturada em quatro blocos temáticos:
* **Bloco A — Mutabilidade e Imutabilidade**: Exercícios 1 e 2
* **Bloco B — Escopo de Variáveis (Global vs Local)**: Exercícios 3 e 4
* **Bloco C — Argumentos Padrão e Calibração Global**: Exercícios 5 e 6
* **Bloco D — Funções Flexíveis (`*args` e `**kwargs`)**: Exercícios 7 e 8

Todas as resoluções comentadas (gabaritos) estão disponíveis na célula imediatamente posterior a cada exercício.

---
### Exercício 1: Validador Seguro de Rotas Submarinas (Imutabilidade e Proteção de Dados)

**Objetivo**: Processar uma lista de waypoints do submarino, garantindo que os dados originais recebidos como entrada **permaneçam imutáveis**, validando cada coordenada geográfica e gerando novas estruturas seguras.

**Instruções**:
Escreva a função `sanitizar_coordenadas_seguro(historico_rotas, limite_lat, limite_lon)` que:
1. Receba `historico_rotas` (uma lista de dicionários no formato `{"id": str, "coordenadas": (lat, lon)}`), `limite_lat` (`float`) e `limite_lon` (`float`).
2. Crie duas listas vazias: `rotas_validas` e `rotas_invalidas`.
3. Percorra a lista `historico_rotas` sem modificar nenhum de seus elementos originais:
   * Desempacote a tupla de coordenadas `(lat, lon)`.
   * Uma rota é válida se `-limite_lat <= lat <= limite_lat` E `-limite_lon <= lon <= limite_lon`.
   * Se for válida, adicione à lista `rotas_validas` um **novo dicionário** contendo `{"id": ponto_id, "coordenadas": coords, "status": "APROVADO"}`.
   * Se for inválida, adicione a tupla `(ponto_id, coords)` à lista `rotas_invalidas`.
4. Crie um dicionário de resumo `metricas` contendo:
   * `"total_processado"`: número total de rotas avaliadas.
   * `"aprovadas"`: quantidade de rotas válidas.
   * `"rejeitadas"`: quantidade de rotas inválidas.
5. Retorne a tupla contendo o dicionário de métricas, a lista de rotas válidas e a lista de rotas inválidas (`return metricas, rotas_validas, rotas_invalidas`).

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 1

def sanitizar_coordenadas_seguro(historico_rotas, limite_lat, limite_lon):
    # Escreva sua lógica aqui!
    pass

# Lista original de coordenadas de navegação
rotas_missao = [
    {"id": "PONTO-01", "coordenadas": (-23.05, -43.15)},
    {"id": "PONTO-02", "coordenadas": (-95.50, -42.00)},  # Latitude inválida (> 90)
    {"id": "PONTO-03", "coordenadas": (-24.10, -44.50)},
    {"id": "PONTO-04", "coordenadas": (-22.80, -195.00)}, # Longitude inválida (> 180)
    {"id": "PONTO-05", "coordenadas": (-23.90, -43.80)}
]

resumo, aprovadas, rejeitadas = sanitizar_coordenadas_seguro(rotas_missao, 90.0, 180.0)

print("Resumo da Validação (Dicionário):", resumo)
print("Rotas Aprovadas (Lista de Novos Dicionários):", aprovadas)
print("Rotas Rejeitadas (Lista de Tuplas):", rejeitadas)
# Resultado esperado:
# Resumo da Validação: {'total_processado': 5, 'aprovadas': 3, 'rejeitadas': 2}
# Rotas Aprovadas: [{'id': 'PONTO-01', 'coordenadas': (-23.05, -43.15), 'status': 'APROVADO'}, {'id': 'PONTO-03', 'coordenadas': (-24.1, -44.5), 'status': 'APROVADO'}, {'id': 'PONTO-05', 'coordenadas': (-23.9, -43.8), 'status': 'APROVADO'}]
# Rotas Rejeitadas: [('PONTO-02', (-95.5, -42.0)), ('PONTO-04', (-22.8, -195.0))]

In [ ]:
# GABARITO DO EXERCÍCIO 1
# def sanitizar_coordenadas_seguro(historico_rotas, limite_lat, limite_lon):
#     rotas_validas = []
#     rotas_invalidas = []
#     
#     for rota in historico_rotas:
#         ponto_id = rota.get("id")
#         coords = rota.get("coordenadas") # tupla (lat, lon)
#         lat, lon = coords
#         
#         if -limite_lat <= lat <= limite_lat and -limite_lon <= lon <= limite_lon:
#             # Criamos um novo dicionário para preservar a imutabilidade do dado de entrada
#             rotas_validas.append({"id": ponto_id, "coordenadas": coords, "status": "APROVADO"})
#         else:
#             rotas_invalidas.append((ponto_id, coords))
#             
#     metricas = {
#         "total_processado": len(historico_rotas),
#         "aprovadas": len(rotas_validas),
#         "rejeitadas": len(rotas_invalidas)
#     }
#     return metricas, rotas_validas, rotas_invalidas

---
### Exercício 2: Gerenciador de Status de Subsistemas (Mutabilidade e Cópia Segura)

**Objetivo**: Atualizar o catálogo de status dos compartimentos do submarino a partir de um relatório de manutenção, utilizando técnicas de cópia explícita de dicionários e listas para garantir que o catálogo original recebido não sofra mutação involuntária por efeito colateral.

**Instruções**:
Escreva a função `registrar_inspecao_submarino(catalogo_subsistemas, novos_status)` que:
1. Receba `catalogo_subsistemas` (dicionário onde a chave é o subsistema e o valor é um dicionário contendo `{"responsavel": str, "status": str, "logs": list}`) e `novos_status` (dicionário `{subsistema: novo_status}`).
2. Crie um novo dicionário vazio `catalogo_atualizado` e uma lista vazia `alteracoes_feitas`.
3. Para cada subsistema e seus dados no catálogo original:
   * Crie uma cópia independente das informações, copiando explicitamente a lista de `logs` com `list(info["logs"])` ou `.copy()`.
   * Se o `subsistema` estiver presente em `novos_status` e o novo status for diferente do atual:
     * Atualize o campo `status` na cópia.
     * Adicione a string `f"Status alterado para {novo_status}"` na lista de `logs` da cópia.
     * Adicione a tupla `(subsistema, status_antigo, novo_status)` à lista `alteracoes_feitas`.
   * Insira o registro copiado/atualizado em `catalogo_atualizado`.
4. Retorne uma tupla contendo o novo catálogo, a lista de alterações e o total de alterações realizadas (`return catalogo_atualizado, alteracoes_feitas, len(alteracoes_feitas)`).

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 2

def registrar_inspecao_submarino(catalogo_subsistemas, novos_status):
    # Escreva sua lógica aqui!
    pass

# Catálogo original de prontidão de bordo
catalogo_base = {
    "Reator": {"responsavel": "Ten. Silva", "status": "OPERACIONAL", "logs": ["Check 08:00 OK"]},
    "Propulsao": {"responsavel": "Sgt. Ramos", "status": "ALERTA", "logs": ["Vibração no eixo"]},
    "Sonar": {"responsavel": "Ten. Costa", "status": "OPERACIONAL", "logs": ["Calibração OK"]}
}

# Atualizações trazidas pela equipe técnica
atualizacoes_manutencao = {
    "Propulsao": "OPERACIONAL",
    "Sonar": "MANUTENCAO"
}

catalogo_novo, alteracoes, total_alt = registrar_inspecao_submarino(catalogo_base, atualizacoes_manutencao)

print("Catálogo Atualizado:", catalogo_novo)
print("Alterações Realizadas (Lista de Tuplas [subsistema, antes, depois]):", alteracoes)
print("Total de Modificações:", total_alt)

# Verificação de segurança de imutabilidade: o catálogo original NÃO deve ter sido modificado!
print("\nLogs originais do Sonar no catálogo base (Deve conter apenas 1 log):", catalogo_base["Sonar"]["logs"])
# Resultado esperado:
# Alterações Realizadas: [('Propulsao', 'ALERTA', 'OPERACIONAL'), ('Sonar', 'OPERACIONAL', 'MANUTENCAO')]
# Total de Modificações: 2

In [ ]:
# GABARITO DO EXERCÍCIO 2
# def registrar_inspecao_submarino(catalogo_subsistemas, novos_status):
#     catalogo_atualizado = {}
#     alteracoes_feitas = []
#     
#     for subsistema, info in catalogo_subsistemas.items():
#         # Cópia segura dos dados e da lista de logs interna
#         info_copia = {
#             "responsavel": info["responsavel"],
#             "status": info["status"],
#             "logs": list(info["logs"])
#         }
#         
#         if subsistema in novos_status:
#             novo_st = novos_status[subsistema]
#             if novo_st != info["status"]:
#                 info_copia["status"] = novo_st
#                 info_copia["logs"].append(f"Status alterado para {novo_st}")
#                 alteracoes_feitas.append((subsistema, info["status"], novo_st))
#                 
#         catalogo_atualizado[subsistema] = info_copia
#         
#     return catalogo_atualizado, alteracoes_feitas, len(alteracoes_feitas)

---
### Exercício 3: Calculadora de Empuxo e Pressão Hidrostática (Escopo Local vs Global)

**Objetivo**: Desenvolver uma rotina de física submarina que utiliza constantes físicas no escopo global para leitura, operando todos os cálculos intermediários em variáveis puramente locais, retornando um relatório consolidado e uma tupla com os limites de pressão.

**Instruções**:
Considere as constantes físicas no escopo global:
* `PRESSAO_ATMOSFERICA_GLOBAL = 1.0` (em atm)
* `DENSIDADE_AGUA_SALGADA_GLOBAL = 1025.0` (em kg/m³)

Escreva a função `calcular_pressao_e_empuxo(leituras_profundidade, volume_submarino)` que:
1. Defina uma variável **local** para a aceleração da gravidade `g = 9.81` (m/s²).
2. Crie listas locais `pressoes_calculadas` e `empuxos_calculados`.
3. Para cada profundidade $h$ (em metros) na lista `leituras_profundidade`:
   * Calcule a pressão hidrostática: $P_{hidro} = \frac{\text{DENSIDADE} \times g \times h}{101325}$ (convertendo Pascal para atm).
   * Calcule a pressão total: $P_{total} = \text{PRESSAO\_ATMOSFERICA\_GLOBAL} + P_{hidro}$, arredondada para 2 casas decimais, e adicione à lista `pressoes_calculadas`.
   * Calcule o empuxo hidrostático: $E = \text{DENSIDADE} \times \text{volume\_submarino} \times g$ (em Newtons), arredondado para 1 casa decimal, e adicione à lista `empuxos_calculados`.
4. Crie um dicionário local `relatorio` contendo:
   * `"pressao_maxima_atm"`: maior pressão calculada.
   * `"pressao_media_atm"`: média das pressões arredondada para 2 casas decimais.
   * `"empuxo_nominal_N"`: valor do empuxo.
5. Retorne o dicionário de relatório, a lista de pressões e uma tupla com a pressão mínima e máxima: `(min(pressoes), max(pressoes))` utilizando retorno múltiplo (`return relatorio, pressoes_calculadas, (min(pressoes_calculadas), max(pressoes_calculadas))`.

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 3

# Constantes Físicas Globais (Somente Leitura)
PRESSAO_ATMOSFERICA_GLOBAL = 1.0      # atm
DENSIDADE_AGUA_SALGADA_GLOBAL = 1025.0 # kg/m³

def calcular_pressao_e_empuxo(leituras_profundidade, volume_submarino):
    # Escreva sua lógica aqui (utilizando variáveis locais para os cálculos)!
    pass

# Teste de imersão operacional
profundidades_teste = [0.0, 50.0, 100.0, 150.0, 200.0] # metros
volume_casco = 1200.0 # m³

relatorio_fisico, lista_p, limites_p = calcular_pressao_e_empuxo(profundidades_teste, volume_casco)

print("Relatório Físico Consolidado (Dicionário):", relatorio_fisico)
print("Pressões por Ponto (Lista):", lista_p)
print("Limites Operacionais (Tupla [min, max]):", limites_p)
# Resultado esperado:
# Relatório Físico: {'pressao_maxima_atm': 20.85, 'pressao_media_atm': 10.92, 'empuxo_nominal_N': 12066480.0}
# Limites Operacionais: (1.0, 20.85)

In [ ]:
# GABARITO DO EXERCÍCIO 3
# def calcular_pressao_e_empuxo(leituras_profundidade, volume_submarino):
#     pressoes_calculadas = []
#     empuxos_calculados = []
#     g = 9.81 # Variável local (aceleração da gravidade)
#     
#     for prof in leituras_profundidade:
#         pressao_hidrostatica = (DENSIDADE_AGUA_SALGADA_GLOBAL * g * prof) / 101325.0
#         pressao_total = round(PRESSAO_ATMOSFERICA_GLOBAL + pressao_hidrostatica, 2)
#         pressoes_calculadas.append(pressao_total)
#         
#         empuxo = round(DENSIDADE_AGUA_SALGADA_GLOBAL * volume_submarino * g, 1)
#         empuxos_calculados.append(empuxo)
#         
#     relatorio = {
#         "pressao_maxima_atm": max(pressoes_calculadas) if pressoes_calculadas else 0.0,
#         "pressao_media_atm": round(sum(pressoes_calculadas) / len(pressoes_calculadas), 2) if pressoes_calculadas else 0.0,
#         "empuxo_nominal_N": empuxos_calculados[0] if empuxos_calculados else 0.0
#     }
#     
#     faixa_pressao = (min(pressoes_calculadas), max(pressoes_calculadas)) if pressoes_calculadas else (0.0, 0.0)
#     return relatorio, pressoes_calculadas, faixa_pressao

---
### Exercício 4: Auditor de Ciclo de Baterias (Gerenciamento de Estado sem Variáveis Globais Mutáveis)

**Objetivo**: Implementar uma função pura de auditoria energética que recebe dados de telemetria de múltiplos bancos de baterias e um limiar crítico, calculando médias por setor em um dicionário, identificando quedas críticas em uma lista de tuplas e retornando uma tupla com o status geral de prontidão, evitando dependência de variáveis globais mutáveis.

**Instruções**:
Escreva a função `auditar_ciclo_energia(historico_baterias, limiar_critico)` que:
1. Receba `historico_baterias` (dicionário `{nome_setor: [leitura1, leitura2, ...]}`) e `limiar_critico` (`float`).
2. Crie localmente um dicionário `resumo_setores`, uma lista `baterias_abaixo_critico` e uma contagem `total_quedas = 0`.
3. Para cada `setor` e sua lista de `niveis`:
   * Calcule a média do setor arredondada para 1 casa decimal e o valor mínimo registrado.
   * Determine o status do setor: `"NORMAL"` se o mínimo for $\ge$ `limiar_critico`, ou `"ALERTA"` caso contrário.
   * Armazene no dicionário `resumo_setores[setor]` o sub-dicionário contendo `{"media": media, "minimo": minimo, "status": status}`.
   * Percorra as leituras do setor com `enumerate(niveis)`. Se algum nível individual for menor que `limiar_critico`, adicione a tupla `(setor, indice_leitura, nivel)` à lista `baterias_abaixo_critico` e incremente `total_quedas`.
4. Determine o status geral do submarino: `"SEGURO"` se `total_quedas == 0`, ou `"CRITICO"` caso contrário.
5. Retorne o dicionário de resumo, a lista de quedas críticas e uma tupla `(status_geral, total_quedas)` com o diagnóstico global (`return resumo_setores, baterias_abaixo_critico, (status_geral, total_quedas)`).

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 4

def auditar_ciclo_energia(historico_baterias, limiar_critico):
    # Escreva sua lógica aqui de forma pura e segura!
    pass

# Histórico percentual de carga dos bancos de bateria por setor
telemetria_energia = {
    "Banco-Principal": [88.5, 84.0, 79.2, 75.0, 68.4],
    "Banco-Emergencia": [98.0, 97.5, 96.0, 95.5, 94.0],
    "Banco-Propulsao": [72.0, 65.5, 58.0, 48.2, 42.0],
    "Banco-Instrumentacao": [91.0, 89.5, 87.0, 84.5, 82.0]
}

resumo_bat, alertas_bat, diagnostico_geral = auditar_ciclo_energia(telemetria_energia, 50.0)

print("Resumo por Setor (Dicionário de Dicionários):", resumo_bat)
print("Quedas Abaixo do Limiar (Lista de Tuplas [setor, ciclo, carga]):", alertas_bat)
print("Diagnóstico Global (Tupla [status, total_quedas]):", diagnostico_geral)
# Resultado esperado:
# Quedas Abaixo do Limiar: [('Banco-Propulsao', 3, 48.2), ('Banco-Propulsao', 4, 42.0)]
# Diagnóstico Global: ('CRITICO', 2)

In [ ]:
# GABARITO DO EXERCÍCIO 4
# def auditar_ciclo_energia(historico_baterias, limiar_critico):
#     baterias_abaixo_critico = []
#     resumo_setores = {}
#     total_quedas = 0
#     
#     for setor, niveis in historico_baterias.items():
#         media_setor = round(sum(niveis) / len(niveis), 1)
#         min_setor = min(niveis)
#         
#         resumo_setores[setor] = {
#             "media": media_setor,
#             "minimo": min_setor,
#             "status": "NORMAL" if min_setor >= limiar_critico else "ALERTA"
#         }
#         
#         for idx, nivel in enumerate(niveis):
#             if nivel < limiar_critico:
#                 baterias_abaixo_critico.append((setor, idx, nivel))
#                 total_quedas += 1
#                 
#     status_geral = "SEGURO" if total_quedas == 0 else "CRITICO"
#     return resumo_setores, baterias_abaixo_critico, (status_geral, total_quedas)

---
### Exercício 5: Calibrador de Sinal com Fator Global (Escopo Global e Leitura)

**Objetivo**: Criar uma função que acesse e utilize uma constante de calibração no escopo global para ajustar um sinal bruto captado pelo sonar de bordo.

**Instruções**:
Considere a constante global `FATOR_CALIBRACAO = 1.25`. Escreva a função `calcular_sinal_ajustado(sinal_bruto)` que:
1. Multiplique o `sinal_bruto` pela variável global `FATOR_CALIBRACAO`.
2. Retorne o valor numérico ajustado.
*Nota: Não declare `FATOR_CALIBRACAO` dentro da função. Mantenha-a no escopo global para simular um parâmetro de calibração central do sonar.*

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 5

# Constante de calibração do sonar central
FATOR_CALIBRACAO = 1.25

def calcular_sinal_ajustado(sinal_bruto):
    # Escreva sua lógica aqui!
    pass

# Testando a função com um sinal bruto de 80.0
sinal_final = calcular_sinal_ajustado(80.0)
print("Sinal ajustado de bordo:", sinal_final)
# Resultado esperado: 100.0

In [ ]:
# GABARITO DO EXERCÍCIO 5
# def calcular_sinal_ajustado(sinal_bruto):
#     return sinal_bruto * FATOR_CALIBRACAO

---
### Exercício 6: Registro Formatado de Mensagens do Sonar (Argumento Padrão)

**Objetivo**: Criar uma função para padronizar mensagens de log com níveis de alerta opcionais utilizando o conceito de argumentos padrão (*default parameters*).

**Instruções**:
Escreva a função `formatar_log(mensagem, nivel="INFO")` que:
1. Receba a string obrigatória `mensagem` e uma string opcional `nivel` (com valor padrão igual a `"INFO"`).
2. Retorne a string formatada no seguinte padrão: `"[NIVEL] MENSAGEM"`, garantindo que o `NIVEL` esteja em letras maiúsculas (`.upper()`).
3. Exemplo: `formatar_log("Bateria em 20%", "perigo")` deve retornar `"[PERIGO] Bateria em 20%"`.
4. Exemplo: `formatar_log("Calibração OK")` deve retornar `"[INFO] Calibração OK"`.

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 6

def formatar_log(mensagem, nivel="INFO"):
    # Escreva sua lógica aqui!
    pass

# Testando chamadas
print(formatar_log("Sistema sonar inicializado"))
# Esperado: [INFO] Sistema sonar inicializado
print(formatar_log("Possível alvo hostil detectado", "alerta"))
# Esperado: [ALERTA] Possível alvo hostil detectado

In [ ]:
# GABARITO DO EXERCÍCIO 6
# def formatar_log(mensagem, nivel="INFO"):
#     return f"[{nivel.upper()}] {mensagem}"

---
### Exercício 7: Média de Leituras Acústicas Variáveis (*args)

**Objetivo**: Criar uma função flexível que aceite qualquer quantidade de leituras acústicas brutas utilizando empacotamento posicional (`*args`) e retorne a média aritmética das leituras.

**Instruções**:
Escreva a função `calcular_media_sinais(*leituras)` que:
1. Receba um número arbitrário de argumentos numéricos com a sintaxe `*leituras`.
2. Se nenhuma leitura for enviada para a função (`len(leituras) == 0`), retorne `0.0`.
3. Caso contrário, calcule a média simples (`sum(leituras) / len(leituras)`).
4. Retorne a média arredondada para duas casas decimais com `round()`.

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 7

def calcular_media_sinais(*leituras):
    # Escreva sua lógica aqui!
    pass

# Testes de chamadas com diferentes números de argumentos
print("Média (3 leituras):", calcular_media_sinais(12.5, 15.0, 10.0))
# Esperado: 12.5
print("Média (nenhuma leitura):", calcular_media_sinais())
# Esperado: 0.0

In [ ]:
# GABARITO DO EXERCÍCIO 7
# def calcular_media_sinais(*leituras):
#     if len(leituras) == 0:
#         return 0.0
#     media = sum(leituras) / len(leituras)
#     return round(media, 2)

---
### Exercício 8: Configurador Dinâmico de Painéis de Controle (**kwargs)

**Objetivo**: Criar uma função de configuração flexível dos subsistemas do submarino a partir de parâmetros variáveis nomeados utilizando `**kwargs`.

**Instruções**:
Escreva a função `configurar_painel(painel_id, **configuracoes)` que:
1. Receba a string `painel_id` posicionalmente e qualquer quantidade de configurações nomeadas com `**configuracoes`.
2. Se nenhuma configuração for fornecida (`len(configuracoes) == 0`), retorne: `"Painel {painel_id} sem configurações adicionais."`
3. Caso haja configurações, percorra o dicionário `configuracoes` e monte uma string com os pares `chave=valor` separados por vírgula e espaço (ex: `"brilho=80, modo=Noturno, som=False"`).
4. Retorne a mensagem formatada: `"Painel {painel_id} configurado com: {config_str}"`.

In [ ]:
# IMPLEMENTAÇÃO DO EXERCÍCIO 8

def configurar_painel(painel_id, **configuracoes):
    # Escreva sua lógica aqui!
    pass

# Testes de chamadas
print(configurar_painel("PAINEL-PROA", brilho=80, modo="Noturno", som=False))
# Esperado: Painel PAINEL-PROA configurado com: brilho=80, modo=Noturno, som=False
print(configurar_painel("PAINEL-MOTOR"))
# Esperado: Painel PAINEL-MOTOR sem configurações adicionais.

In [ ]:
# GABARITO DO EXERCÍCIO 8
# def configurar_painel(painel_id, **configuracoes):
#     if len(configuracoes) == 0:
#         return f"Painel {painel_id} sem configurações adicionais."
#     config_str = ", ".join([f"{k}={v}" for k, v in configuracoes.items()])
#     return f"Painel {painel_id} configurado com: {config_str}"